# IEEE‑CIS Fraud Detection — High‑Performance Modeling Pipeline

In [2]:
!pip install lightgbm
!pip install xgboost
!pip install catboost
!pip install tensorflow
!pip install scikit-learn
!pip install pandas numpy
!pip install catboost[gpu]
!pip install jax jaxlib flax


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 52.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 39.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 30.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 95.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] 

# 1. Import Framework

In [3]:
#load libraries
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# 2. Load Data

In [4]:
#Load the Data
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity     = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


# 3. Merge Data

In [5]:
#Merge the Data
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")

# 4. Memory Optimizatio

In [6]:
def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype

        # Skip string/arrow columns safely
        if pd.api.types.is_string_dtype(col_type):
            df[col] = df[col].astype("category")
            continue

        # Skip categorical columns
        if pd.api.types.is_categorical_dtype(col_type):
            continue

        # Numeric columns
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)

        else:
            # fallback for anything unexpected
            df[col] = df[col].astype("category")

    return df


# 5. Split Data into Features and Target

In [7]:
y = train["isFraud"]
X = train.drop("isFraud", axis=1)
X_test = test.copy()

# 6. Align Train And Test Columns

In [8]:
#Align columns between train and test
missing_cols = [col for col in X.columns if col not in X_test.columns]
X_test = pd.concat([X_test, pd.DataFrame(np.nan, index=X_test.index, columns=missing_cols)], axis=1)
X_test = X_test[X.columns]


# 7. Feature Engineering 

In [9]:
new = {}
new_test = {}

# --- Time features ---
new["day"] = X["TransactionDT"] // (24 * 60 * 60)
new_test["day"] = X_test["TransactionDT"] // (24 * 60 * 60)

new["hour"] = (X["TransactionDT"] // 3600) % 24
new_test["hour"] = (X_test["TransactionDT"] // 3600) % 24

# --- Amount features ---
new["log_amt"] = np.log1p(X["TransactionAmt"])
new_test["log_amt"] = np.log1p(X_test["TransactionAmt"])

new["amt_per_day"] = X["TransactionAmt"] / (new["day"] + 1)
new_test["amt_per_day"] = X_test["TransactionAmt"] / (new_test["day"] + 1)

# --- Frequency encoding ---
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]
for col in freq_cols:
    freq = X[col].value_counts()
    new[col + "_freq"] = X[col].map(freq)
    new_test[col + "_freq"] = X_test[col].map(freq)

# --- Device features ---
new["device_name"] = X["DeviceInfo"].str.split("/", expand=True)[0]
new_test["device_name"] = X_test["DeviceInfo"].str.split("/", expand=True)[0]

new["device_version"] = X["DeviceInfo"].str.split("/", expand=True)[1]
new_test["device_version"] = X_test["DeviceInfo"].str.split("/", expand=True)[1]

# --- Email domain grouping ---
email_map = {
    "gmail.com": "google", "gmail": "google",
    "yahoo.com": "yahoo", "yahoo.com.mx": "yahoo",
    "hotmail.com": "microsoft", "outlook.com": "microsoft"
}
new["P_emaildomain_group"] = X["P_emaildomain"].map(email_map)
new_test["P_emaildomain_group"] = X_test["P_emaildomain"].map(email_map)

# --- UID features ---
X["uid"] = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
X_test["uid"] = X_test["card1"].astype(str) + "_" + X_test["addr1"].astype(str)

X["uid2"] = X["uid"] + "_" + X["card2"].astype(str)
X_test["uid2"] = X_test["uid"] + "_" + X_test["card2"].astype(str)

# --- Aggregations on TransactionAmt ---
for col in ["uid", "card1", "card2", "P_emaildomain"]:
    X[f"{col}_amt_mean"] = X.groupby(col)["TransactionAmt"].transform("mean")
    X_test[f"{col}_amt_mean"] = X_test.groupby(col)["TransactionAmt"].transform("mean")

# --- Count encoding ---
for col in ["card1", "card2", "addr1", "P_emaildomain"]:
    freq = X[col].value_counts()
    X[col + "_count"] = X[col].map(freq)
    X_test[col + "_count"] = X_test[col].map(freq)

# Add engineered features to X and X_test
X = pd.concat([X, pd.DataFrame(new)], axis=1)
X_test = pd.concat([X_test, pd.DataFrame(new_test)], axis=1)


/tmp/ipykernel_14/721241319.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["uid"] = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
/tmp/ipykernel_14/721241319.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["uid2"] = X["uid"] + "_" + X["card2"].astype(str)
/tmp/ipykernel_14/721241319.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To 

# 8. Encode categorical features

In [10]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))



/tmp/ipykernel_14/2572361007.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns


# 9. Automatic Column Detection

In [11]:
# Identify numeric and categorical columns:
numeric_cols = X.select_dtypes(include=["int", "float", "float32", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns

# 10. Convert Numeric Columns To Float32

In [12]:
X[numeric_cols] = X[numeric_cols].astype("float32")
X_test[numeric_cols] = X_test[numeric_cols].astype("float32")

# 11. Handle missing values

In [13]:
#Handle Missing Values
num_cols = X.select_dtypes(include=["int", "float"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

X[num_cols] = X[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

X[cat_cols] = X[cat_cols].fillna("missing")
X_test[cat_cols] = X_test[cat_cols].fillna("missing")

X = X.fillna(-1)
X_test = X_test.fillna(-1)


# 12.Fill Missing Values

In [14]:
train_sorted = train.sort_values("TransactionDT")
idx = train_sorted.index

X_sorted = X.loc[idx]
y_sorted = y.loc[idx]

split = int(len(X_sorted) * 0.8)

X_train = X_sorted.iloc[:split]
y_train = y_sorted.iloc[:split]

X_valid = X_sorted.iloc[split:]
y_valid = y_sorted.iloc[split:]

# 18. Train And Validation Split

# 13. Lightgbm Model

In [15]:
train_set = lgb.Dataset(X_train, y_train)
valid_set = lgb.Dataset(X_valid, y_valid)

params_lgb = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.005,
    "num_leaves": 512,
    "max_depth": -1,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "min_data_in_leaf": 50,
    "max_bin": 255,
    "verbosity": -1
}

model_lgb = lgb.train(
    params_lgb,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=40000,
    callbacks=[lgb.early_stopping(500), lgb.log_evaluation(200)]
)

valid_pred_lgb = model_lgb.predict(X_valid)
auc_lgb = roc_auc_score(y_valid, valid_pred_lgb)
print("LightGBM AUC:", auc_lgb)

Training until validation scores don't improve for 500 rounds
[200]	training's auc: 0.956513	valid_1's auc: 0.899518
[400]	training's auc: 0.975794	valid_1's auc: 0.91343
[600]	training's auc: 0.989603	valid_1's auc: 0.923289
[800]	training's auc: 0.995628	valid_1's auc: 0.928294
[1000]	training's auc: 0.998378	valid_1's auc: 0.930733
[1200]	training's auc: 0.999409	valid_1's auc: 0.932024
[1400]	training's auc: 0.99979	valid_1's auc: 0.932927
[1600]	training's auc: 0.999923	valid_1's auc: 0.93345
[1800]	training's auc: 0.999973	valid_1's auc: 0.93389
[2000]	training's auc: 0.99999	valid_1's auc: 0.934332
[2200]	training's auc: 0.999997	valid_1's auc: 0.934619
[2400]	training's auc: 0.999999	valid_1's auc: 0.934813
[2600]	training's auc: 1	valid_1's auc: 0.934813
[2800]	training's auc: 1	valid_1's auc: 0.934912
[3000]	training's auc: 1	valid_1's auc: 0.93497
[3200]	training's auc: 1	valid_1's auc: 0.935033
[3400]	training's auc: 1	valid_1's auc: 0.934981
[3600]	training's auc: 1	valid_

# 14. XGBoost model

In [16]:
model_xgb = xgb.XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    tree_method="hist"
)

model_xgb.fit(X_train, y_train)

valid_pred_xgb = model_xgb.predict_proba(X_valid)[:, 1]

auc_xgb = roc_auc_score(y_valid, valid_pred_xgb)
print("XGBoost AUC:", auc_xgb)


XGBoost AUC: 0.9282646125480095


# 15. CatBoost model

In [17]:
cat_features_idx = []

cat_model = CatBoostClassifier(
    iterations=4000,
    learning_rate=0.03,
    depth=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_strength=1.5,
    l2_leaf_reg=3,
    border_count=128,
    verbose=False
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features_idx
)

valid_pred_cat = cat_model.predict_proba(X_valid)[:, 1]
auc_cat = roc_auc_score(y_valid, valid_pred_cat)
print("CatBoost AUC:", auc_cat)

CatBoost AUC: 0.9221474419874414


# 17. CatBoost Block Improved + Tuned

In [19]:
cat_model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_strength=1.0,
    l2_leaf_reg=3,
    border_count=64,
    task_type="CPU",
    verbose=200
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=200
)

valid_pred_cat = cat_model.predict_proba(X_valid)[:, 1]
auc_cat = roc_auc_score(y_valid, valid_pred_cat)
print("CatBoost AUC:", auc_cat)

0:	test: 0.6834921	best: 0.6834921 (0)	total: 63ms	remaining: 1m 15s
200:	test: 0.8753554	best: 0.8753554 (200)	total: 12.1s	remaining: 1m
400:	test: 0.8875062	best: 0.8875062 (400)	total: 24.3s	remaining: 48.4s
600:	test: 0.8948044	best: 0.8948044 (600)	total: 36.4s	remaining: 36.3s
800:	test: 0.8991500	best: 0.8991500 (800)	total: 48.3s	remaining: 24s
1000:	test: 0.9029931	best: 0.9029931 (1000)	total: 1m	remaining: 12s
1199:	test: 0.9058829	best: 0.9058834 (1198)	total: 1m 12s	remaining: 0us

bestTest = 0.9058833868
bestIteration = 1198

Shrink model to first 1199 iterations.
CatBoost AUC: 0.9058833867685272


# 18. Stacking / blending

In [30]:
stack_valid = (
    0.55 * valid_pred_lgb +
    0.25 * valid_pred_xgb +
    0.15 * valid_pred_cat
)

auc_stack = roc_auc_score(y_valid, stack_valid)
print("Stacked AUC:", auc_stack)

Stacked AUC: 0.9346811909624881


# 19 Retrain LightGBM on full data

In [31]:
full_set = lgb.Dataset(X, y)

model_lgb_full = lgb.train(
    params_lgb,
    full_set,
    num_boost_round=model_lgb.best_iteration
)

# 20. Test predictions

In [32]:
test_pred_lgb = model_lgb_full.predict(X_test)
test_pred_xgb = model_xgb.predict_proba(X_test)[:, 1]
test_pred_cat = cat_model.predict_proba(X_test)[:, 1]

test_pred_stack = (
    0.55 * test_pred_lgb +
    0.25 * test_pred_xgb +
    0.15 * test_pred_cat 

)

# 21. Create Submission File

In [34]:

submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_pred_stack
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


In [35]:
check = pd.read_csv("submission.csv")
check.head()


,TransactionID,isFraud
0,3663549,0.000397
1,3663550,0.001679
2,3663551,0.001873
3,3663552,0.000710
4,3663553,0.001273
